In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from torch.utils.data import Dataset,DataLoader


In [2]:
df=pd.read_csv('WELFake_Dataset.csv.zip')

In [3]:
df.shape
df.columns.to_list()
df.head()
df.isnull().sum()

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

In [4]:
df['label'].value_counts()

label
1    37106
0    35028
Name: count, dtype: int64

In [5]:
df['title'] = df['title'].fillna("")
df['text'] = df['text'].fillna("")

In [6]:
df.isnull().sum()


Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [7]:
df['combined'] = df['title'] + " " + df['text']

In [8]:
df['combined'].head()

0    LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1       Did they post their votes for Hillary already?
2    UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3    Bobby Jindal, raised Hindu, uses story of Chri...
4    SATAN 2: Russia unvelis an image of its terrif...
Name: combined, dtype: str

In [9]:
df['combined'] = df['combined'].str.replace(r'\s+', ' ', regex=True)

In [10]:
df['combined'].head()

0    LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1       Did they post their votes for Hillary already?
2    UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3    Bobby Jindal, raised Hindu, uses story of Chri...
4    SATAN 2: Russia unvelis an image of its terrif...
Name: combined, dtype: str

In [11]:
df.drop_duplicates(subset='combined', inplace=True)

In [12]:
df.shape

(63675, 5)

In [13]:
df['label'].value_counts()

label
0    34790
1    28885
Name: count, dtype: int64

In [14]:
df.drop(columns=['Unnamed: 0', 'title', 'text'], inplace=True)

In [15]:
df.columns.to_list()

['label', 'combined']

In [16]:
df.shape

(63675, 2)

In [17]:
df.to_csv('cleaned_data.csv', index=False)

In [18]:
df=pd.read_csv('cleaned_data.csv')

In [19]:
train_df, temp_df  = train_test_split(df, test_size=0.3, random_state=42)

In [20]:
val_df,test_df=train_test_split(temp_df,test_size=0.5,random_state=42)

In [21]:
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 44572
Validation: 9551
Test: 9552


In [22]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [23]:
sample = "This news is completely fake"
tokens = tokenizer(sample, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
print(tokens)
print(tokens['input_ids'].shape)

{'input_ids': tensor([[ 101, 2023, 2739, 2003, 3294, 8275,  102,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,

In [25]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
      text = self.texts.iloc[idx]
      encoding = self.tokenizer(
      text,
      max_length=self.max_length,
      truncation=True,
      padding='max_length',
      return_tensors='pt'
      )
      input_ids= encoding['input_ids'].squeeze()
      attention_mask=encoding['attention_mask'].squeeze()
      label=self.labels.iloc[idx]
      return input_ids, attention_mask, label

In [26]:
train_dataset = FakeNewsDataset(
    texts=train_df['combined'],
    labels=train_df['label'],
    tokenizer=tokenizer
)
test_dataset = FakeNewsDataset(
    texts=test_df['combined'],
    labels=test_df['label'],
    tokenizer=tokenizer
)
val_dataset = FakeNewsDataset(
    texts=val_df['combined'],
    labels=val_df['label'],
    tokenizer=tokenizer
)

In [27]:
print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Train: 44572
Val: 9551
Test: 9552


In [29]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [30]:
batch = next(iter(train_loader))
input_ids, attention_mask, labels = batch
print(f"input_ids shape: {input_ids.shape}")
print(f"attention_mask shape: {attention_mask.shape}")
print(f"labels shape: {labels.shape}")

input_ids shape: torch.Size([32, 512])
attention_mask shape: torch.Size([32, 512])
labels shape: torch.Size([32])
